# Task 1: Load and Inspect the Data
 We check the structure of the data and identify the count of null values in each column.

In [3]:
import pandas as pd
import numpy as np
data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                        'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                        'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                        'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                    560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                    33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                      'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                      'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                      'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)
df.info()
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB
transaction_id      0
date                0
region              3
product_category    2
sales_amount        4
quantity            3
customer_age        4
payment_method      3
dtype: int64


## Task 2: Handle Missing Values
In this step, we apply various cleaning strategies

In [4]:
df['region'] = df['region'].fillna(df['region'].mode()[0])
df['product_category'] = df['product_category'].fillna(df['product_category'].mode()[0])

df['sales_amount'] = df['sales_amount'].fillna(df['sales_amount'].median())

df['quantity'] = df['quantity'].ffill()

df['customer_age'] = df['customer_age'].fillna(round(df['customer_age'].mean()))

df.dropna(subset=['payment_method'], inplace=True)

print("\n--- Missing Values After Cleaning ---")
print(df.isna().sum())


--- Missing Values After Cleaning ---
transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64


## Task 3: GroupBy Analysis
Aggregating the data to find total sales by region and identifying the highest performing product-region combinations.

In [5]:
total_sales_region = df.groupby('region')['sales_amount'].sum()
print("\n--- Total Sales by Region ---")
print(total_sales_region)

avg_sales_category = df.groupby('product_category')['sales_amount'].mean()
print("\n--- Average Sales by Category ---")
print(avg_sales_category)

grouped_data = df.groupby(['region', 'product_category'])[['sales_amount', 'quantity']].sum().reset_index()

top_3_combinations = grouped_data.sort_values(by='sales_amount', ascending=False).head(3)
print("\n--- Top 3 Region-Product Combinations ---")
print(top_3_combinations)


--- Total Sales by Region ---
region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64

--- Average Sales by Category ---
product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64

--- Top 3 Region-Product Combinations ---
  region product_category  sales_amount  quantity
5  North             Home        3250.0      12.0
4  North      Electronics        2700.0       3.0
2   East      Electronics        2100.0       1.0


## Task 4: Custom Aggregation
Applying a custom function to calculate the **Sales Range** (Max - Min) and generating a summary table using `.agg()`.

In [6]:
def sales_range(x):
    return x.max() - x.min()

region_range = df.groupby('region')['sales_amount'].apply(sales_range)
print("\n--- Sales Range by Region ---")
print(region_range)

multi_metrics = df.groupby('region').agg({
    'sales_amount': ['sum', 'mean', 'max'],
    'quantity': ['sum', 'min']
})
print("\n--- Multiple Metrics by Region ---")
print(multi_metrics)


--- Sales Range by Region ---
region
East     1720.0
North     990.0
South     275.0
West       55.0
Name: sales_amount, dtype: float64

--- Multiple Metrics by Region ---
       sales_amount                     quantity     
                sum        mean     max      sum  min
region                                               
East         3790.0  947.500000  2100.0      8.0  1.0
North        6460.0  922.857143  1500.0     18.0  1.0
South        1900.0  633.333333   725.0      9.0  2.0
West         2230.0  743.333333   780.0      7.0  1.0
